In [43]:
%matplotlib widget

In [44]:
%pip install openpmd_viewer openpmd-api

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [97]:
from matplotlib import pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
import openpmd_viewer as ov
from openpmd_viewer import OpenPMDTimeSeries
import os

path = os.path.join(os.environ['PSCRATCH'], 'diags', 'test')
print(path)
ts_2d = OpenPMDTimeSeries(path)


    
    



/pscratch/sd/l/leosyam/diags/test


In [106]:
angs = np.linspace(0, np.pi, 200)
vecs = []
perpvecs = []
for i in angs:
    vecs.append([np.cos(i), np.sin(i)])
    perpvecs.append([-np.sin(i), np.cos(i)])
vecs = np.array(vecs).T
perpvecs = np.array(perpvecs).T
def loopfrompoints(pts, range_percent, close_percent, conv_percent=1):
    center = np.mean(pts, axis=0)
    pts2 = pts - center
    stdevx = np.std(pts2[:, 0])
    stdevy = np.std(pts2[:, 1])
    pts2[:, 0] /= stdevx
    pts2[:, 1] /= stdevy
    dots = np.matmul(pts2, vecs)
    dists = np.square(np.matmul(pts2, perpvecs))
    idxs = np.argsort(dists, axis=0)
    dots = dots[idxs, np.arange(dots.shape[1])][:int(close_percent * 0.01 * dots.shape[0])]
    dots.sort(axis=0) 
    mid = dots.shape[0] // 2
    off = int((dots.shape[0] * range_percent * 0.01) / 2)
    l = mid - off
    if l < 0:
        l = 0
    r = mid + off
    if r >= dots.shape[0]:
        r = dots.shape[0] - 1
    set1 = vecs * dots[r]
    set2 = vecs * dots[l]
    set2 = np.append(set2, set1[:, [0]], axis=1)
    loop = np.append(set1, set2, axis=1).T
    smoothloop = np.empty(shape=loop.shape)
    convnum = int(conv_percent * 0.01 * loop.shape[0])
    if convnum > 0:    
        for i in range(len(loop)):
            xsum = 0
            ysum = 0
            for j in range(convnum):
                xsum += loop[i-j, 0]
                ysum += loop[i-j, 1]
            xsum /= convnum
            ysum /= convnum
            smoothloop[i - (convnum//2), 0] = xsum
            smoothloop[i - (convnum//2), 1] = ysum
    else:
        smoothloop = loop
    smoothloop[:, 0] *= stdevx
    smoothloop[:, 1] *= stdevy
    smoothloop = np.append(smoothloop, smoothloop[0][np.newaxis, :], axis=0)
    return smoothloop + center
    

plt.close('all')
import cv2

video = None
first = True
xlims = (np.float64(-4.3473773124785324e-05), np.float64(4.373071461582156e-05))
ylims = (np.float64(-1.4656088338057575e-05), np.float64(1.4475673383299323e-05))

for it in range(0, 500, 5):
    points = np.array(ts_2d.get_particle(species='test1', var_list=['x', 'y'], iteration=it)).T
    loop = loopfrompoints(points, 95, 25, conv_percent=8)

    fig, ax = plt.subplots()
    ax.scatter(points[:, 0], points[:, 1], s=1)
    ax.plot(loop[:, 0], loop[:, 1])

    # ax.set_xlim(*xlims)
    # ax.set_ylim(*ylims)
        

    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())

    frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)    
    
    if first:
        height, width, _ = frame.shape
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video = cv2.VideoWriter('contour_points_vid_cameramove.mp4', fourcc, 15, (width, height))
        video.write(frame)
        first = False
    else:
        video.write(frame)

    plt.close(fig)


video.release()

In [105]:
!ls

claude_shape.py		       rectangle_crosssections_plot.ipynb
contour_points_vid.mp4	       shapeof2dpoints_failed2.py
failed_circle_algorithm.ipynb  shapeof2dpoints_normedfirst.py
gemini_shape.py		       shapeof2dpoints.py
point_contours.ipynb	       Untitled.ipynb
